# Заняття 6 · Демо 3 · Grounded-SAM 2 на відео: текст → бокси → маски → треки

**LIVE DEMO 3 (слайд 55)** · слайди 26–32, 44–54, 57–63

Зшиваємо вузли демо 1 і 2 в один ланцюг (слайд 52) і проганяємо його на справжньому бортовому відео:
**VisDrone2019-VID-val, `uav0000268_05773_v`** — 4K, камера летить уперед, машина з першого плану від'їжджає й ховається
за лісосмугою. Покадрова розмітка VisDrone з ID треків і рівнем перекриття дає те, чого зазвичай немає на лекції:
**чесні цифри** для мерехтіння, втрати ідентичності, оклюзій і корекції.

```
ТЕКСТ ──► GROUNDING DINO ──► ФІЛЬТР / NMS ──► SAM 2.1 ──► SAM 2 propagate ──► ЕКСПОРТ
промпт    бокси + score     найвразливіший    бокс → маска   пам'ять, оклюзії     маски, кошики, канал
```

| Розділ | Що показуємо | Слайди |
|---|---|---|
| 1. Ключовий кадр: текст → бокси | 4K → дрібні цілі; повний кадр проти тайлів; P / R за GT | 44–46, 50 |
| 2. Фільтр боксів | Найвразливіший вузол: бокс на кущ → SAM слухняно дає маску | 52 |
| 3. SAM покадрово проти SAM 2 | Мерехтіння площі, перескоки, «домальовування» за перешкодою | 27, 28 |
| 4. Оклюзія, дрейф і корекція | Модель руху (Калман) знаходить підозрілі кадри, один бокс виправляє решту ролика | 28–30 |
| 5. Скільки це коштує | Вузли, T_frame, SAM 2 проти «детектор + ByteTrack» від n_obj | 24, 31, 53 |
| 6. **Каркас ДЗ** | Увесь пайплайн однією функцією + експорт масок, вирізань, COCO RLE | 63 |
| 7. Машина розмітки | Три кошики «прийняти / перевірити / відкинути», бюджет каналу маска vs бокс | 57, 58, 60 |
| 8. ІЧ: геометрія переживає домен | Текст на LWIR мовчить, клік на LWIR працює | 8, 47 |
| 9. Композиція чи одна модель | SAM 3 проти Grounding DINO + SAM 2.1 на тому самому кадрі (якщо є ваги) | 32, 54 |

**Підготовка (один раз, з мережею):**
```bash
# з кореня репозиторію
python tools/fetch_visdrone.py --splits val                                   # VisDrone2019-VID-val, ~1.5 ГБ
python tools/fetch_sam_models.py --models sam2.1-small                        # + sam3 — gated, для розділу 9
python tools/fetch_ov_models.py --models gdino-base yolo11
```
Увесь ноутбук — 3–5 хвилин на GPU. Кадри кліпу кешуються в `outputs/demo3/frames/`, експорт — в `outputs/demo3/export/`.

In [ ]:
from pathlib import Path

# ---------------- Конфігурація ----------------
DEVICE = None
SEQ, START, N_FRAMES, STRIDE = "uav0000268_05773_v", 41, 150, 2   # кадри 41…339 кожен другий: 10 с польоту
MAX_SIDE = 1920                # кліп зменшуємо з 4K: SAM 2 однаково бачить 1024, а маски 4K не влізуть у RAM
GDINO, SAM2 = "gdino-base", "sam2.1-small"
PROMPT = "car . truck . bus . van ."
CLASSES = ["car", "truck", "bus", "van"]
BOX_THR, TEXT_THR = 0.25, 0.25
TILE, OVERLAP = 640, 0.2
MAX_OBJ = 16                   # SAM 2: кожен об'єкт — окрема вартість
SEED = 0
OUT = Path("outputs/demo3")

import panels as P
USE_SAM3 = P.have("sam3")
hf = P.setup_hf(sam=[SAM2] + (["sam3"] if USE_SAM3 else []), ov=[GDINO, "yolo11"])

import json, time
import numpy as np, pandas as pd, torch, cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
pd.set_option("display.precision", 3)
DEVICE = P.pick_device(DEVICE)
GPU = P.gpu_name(DEVICE)
OUT.mkdir(parents=True, exist_ok=True)

seqs = P.vid_sequences()
clip = P.make_clip(SEQ, START, N_FRAMES, STRIDE, OUT / "frames", MAX_SIDE)
ann = P.vid_annotations(SEQ)
gt = P.clip_gt(clip, ann)                       # GT у координатах кліпу: k, tid, бокс, occ
W, H = clip.size
key = clip.read(0)
src = seqs.set_index("seq").loc[SEQ]
print(f"HF: {hf} | {DEVICE}: {GPU} | {SEQ}: {src.W}×{src.H}, кліп {len(clip.frame_ids)} кадрів {W}×{H} "
      f"(×{clip.scale:.2f}) | треків машин у кліпі: {gt.tid.nunique()}")

## 1. Ключовий кадр: текст → бокси

Grounding DINO на першому кадрі кліпу — повним кадром і тайлами 640 / 20 % (слайд 46). Порівнюємо з GT VisDrone цього
кадру: precision / recall у робочій точці `BOX_THR` (слайд 50). Машини в глибині кадру — 20–40 px: для повного кадру
це вже межа (слайд 45).

In [ ]:
gd = P.GDino(GDINO, DEVICE)
detect_full = lambda im: gd(im, PROMPT, CLASSES, box_thr=BOX_THR, text_thr=TEXT_THR)
g0 = gt[gt.k == 0]
gt0 = g0[["x0", "y0", "x1", "y1"]].values

t_full, t_tile = [], []
with P.timed(t_full):
    d_full = detect_full(key)
with P.timed(t_tile):
    d_tile, n_pass = P.detect_tiled(detect_full, key, TILE, OVERLAP, with_full=True)
rows = []
for name, d, t in [("повний кадр", d_full, t_full[0]), (f"тайли {TILE} ×{n_pass - 1} + повний", d_tile, t_tile[0])]:
    r = P.match_pr(d.boxes, gt0)
    rows.append({"режим": name, "боксів": len(d), "TP": r["tp"], "FP": r["fp"], "пропущено": r["fn"],
                 "precision": r["tp"] / max(len(d), 1), "recall": r["tp"] / len(gt0), "мс": t})
display(pd.DataFrame(rows).set_index("режим"))
side = np.maximum(gt0[:, 2] - gt0[:, 0], gt0[:, 3] - gt0[:, 1])
print(f"GT-машин на кадрі: {len(gt0)}, сторона {side.min():.0f}–{side.max():.0f} px кліпу "
      f"(на вході GDINO ×{min(800 / H, 1333 / W):.2f} → {side.min() * min(800 / H, 1333 / W):.0f}–{side.max() * min(800 / H, 1333 / W):.0f} px)")

# машини займають малу частину кадру — показуємо збільшену ділянку навколо них
central = gt0[((gt0[:, 0] + gt0[:, 2]) / 2 > 0.2 * W) & ((gt0[:, 0] + gt0[:, 2]) / 2 < 0.8 * W)]
ROI = np.r_[np.maximum(central[:, :2].min(0) - 80, 0), np.minimum(central[:, 2:].max(0) + 80, [W, H])].astype(int)
roi = lambda im: im[ROI[1]:ROI[3], ROI[0]:ROI[2]]
vis = [roi(P.draw_boxes(P.draw_boxes(key, gt0, color=(150, 150, 160), thick=3), d.boxes, [f"{s:.2f}" for s in d.scores],
                        thick=2, font=0.5)) for d in (d_full, d_tile)]
P.show_row(vis, [r["режим"] for r in rows], size=7, suptitle="сірі — GT VisDrone, бірюзові — Grounding DINO (збільшена ділянка)",
           save=OUT / "03_keyframe_full_vs_tiles.png")

## 2. Фільтр боксів — найвразливіший вузол

SAM не має механізму сказати «цей бокс безглуздий» — він слухняно побудує маску навколо будь-чого (слайд 52). Тому між
детектором і SAM стоїть фільтр, і саме його пороги — головна ручка якості датасету. Нижче він написаний повністю:
мінімальна сторона, максимальна частка кадру, score, NMS між класами, **вкладені бокси** і ліміт об'єктів.

Вкладені бокси — побічний ефект тайлів: машина на межі тайла в сусідньому тайлі видна наполовину, і детектор дає їй
окремий «пів-бокс». NMS його не прибирає (IoU половини з цілим < 0.5), а SAM 2 потім сумлінно веде «пів машини» весь ролик.
Тому порівнюємо ще й **перетин / площа меншого** боксу (IoS).

In [ ]:
def filter_boxes(d, min_side=8, max_area_frac=0.05, min_score=0.3, nms_iou=0.6, contain=0.8, max_n=MAX_OBJ):
    w, h = d.boxes[:, 2] - d.boxes[:, 0], d.boxes[:, 3] - d.boxes[:, 1]
    ok = (np.minimum(w, h) >= min_side) & (w * h <= max_area_frac * W * H) & (d.scores >= min_score) & (d.labels >= 0)
    d = P.nms(d.take(np.flatnonzero(ok)), nms_iou, class_aware=False)    # «car» і «van» на одній машині — один об'єкт
    # від більших боксів до менших: менший, що майже весь лежить у вже прийнятому, — частина об'єкта
    keep = []
    for i in np.argsort(-(d.boxes[:, 2] - d.boxes[:, 0]) * (d.boxes[:, 3] - d.boxes[:, 1])):
        if not keep or P.box_ios(d.boxes[i], d.boxes[keep]).max() < contain:
            keep.append(i)
    d = d.take(np.array(keep, int))
    return d.take(np.argsort(-d.scores)[:max_n])


boxes_no_ios = filter_boxes(d_tile, contain=1.01)                      # той самий фільтр без перевірки вкладеності
boxes0 = filter_boxes(d_tile)
for name, b in [("до фільтра", d_tile), ("фільтр без IoS", boxes_no_ios), ("повний фільтр", boxes0)]:
    r = P.match_pr(b.boxes, gt0)
    print(f"{name:15s}: {len(b):2d} боксів, TP {r['tp']}, FP {r['fp']}, recall {r['tp']}/{len(gt0)}")

# «поганий» бокс: ділянка лісосмуги, де немає жодної машини
img_pred = P.load_sam2_image(SAM2, DEVICE)
with torch.inference_mode(), P.autocast(DEVICE):
    img_pred.set_image(key)
    m_good, iou_good, _ = img_pred.predict(box=boxes0.boxes, multimask_output=False)
    cand = [(x, y) for y in range(0, H - 100, 50) for x in range(0, W - 160, 50)]
    far = lambda c: np.min(np.hypot((gt0[:, 0] + gt0[:, 2]) / 2 - c[0] - 80, (gt0[:, 1] + gt0[:, 3]) / 2 - c[1] - 50))
    bx, by = max(cand, key=far)
    bad_box = np.array([bx, by, bx + 160, by + 100], np.float32)
    m_bad, iou_bad, _ = img_pred.predict(box=bad_box, multimask_output=False)
m_good = m_good.reshape(len(boxes0), H, W) > 0
print(f"predicted_iou: машини {np.round(iou_good.ravel(), 2)} | бокс на фон {iou_bad.ravel()[0]:.2f}")
vis_bad = P.draw_boxes(P.overlay_masks(key, [m_bad[0] > 0], alpha=.6), bad_box, color=(240, 70, 70))
half = [i for i in range(len(boxes_no_ios)) if P.box_ios(boxes_no_ios.boxes[i], boxes0.boxes).max() > 0.8
        and P.box_iou(boxes_no_ios.boxes[i], boxes0.boxes).max() < 0.9]
vis_half = P.draw_boxes(P.draw_boxes(key, boxes0.boxes, thick=2), boxes_no_ios.boxes[half], color=(240, 70, 70), thick=2)
for x in sorted({x for x, _ in P.tile_origins(W, H, TILE, OVERLAP)} | {x + TILE for x, _ in P.tile_origins(W, H, TILE, OVERLAP)}):
    vis_half[:, min(x, W - 1)] = (255, 220, 0)                           # межі тайлів по x
for y in sorted({y for _, y in P.tile_origins(W, H, TILE, OVERLAP)} | {y + TILE for _, y in P.tile_origins(W, H, TILE, OVERLAP)}):
    vis_half[min(y, H - 1)] = (255, 220, 0)
bad_crop, _ = P.crop_around(vis_bad, bad_box, pad=120)
P.show_row([roi(P.overlay_masks(P.draw_boxes(key, boxes0.boxes, thick=2), m_good)), roi(vis_half), bad_crop],
           ["бокси після фільтра → маски SAM 2.1", "червоні — «пів-бокси» на межах тайлів (жовті лінії)",
            f"бокс на фон → маска, predicted_iou {iou_bad.ravel()[0]:.2f}"],
           size=5.5, save=OUT / "03_bad_box_poisons_sam.png")

**Підсумок розділів 1–2.** На 4K-кадрі, стиснутому до входу GDINO, дальні машини губляться — тайли повертають їх ціною
×`n_pass` проходів і приносять власні артефакти: хибні спрацювання та «пів-бокси» на межах тайлів. Сміття прибирає фільтр,
а не SAM: `predicted_iou` у маски «куща» того самого порядку, що в машини (демо 1, розділ 5). Помилка детектора, яку
пропустив фільтр, мовчки стає помилкою в датасеті — на кожному кадрі ролика.

## 3. SAM покадрово проти SAM 2 з пам'яттю

Обидва варіанти стартують з тих самих боксів на кадрі 0 (слайд 27):
* **покадрово:** на кожному кадрі SAM 2.1 *image* отримує бокс маски з попереднього кадру — пам'яті немає, кожен кадр незалежно;
* **SAM 2:** `propagate_in_video` з банком пам'яті й головою оклюзії (слайд 28).

Кожен об'єкт зіставляємо з GT-треком на кадрі 0 і міряємо покадрово: IoU бокса маски з «своїм» GT там, де об'єкт видно;
чи є маска там, де GT каже «сильно перекрито» або об'єкт уже пішов; чи маска перескочила на сусідню машину.

In [ ]:
# ---- SAM 2 з пам'яттю ----
vp = P.load_sam2_video(SAM2, DEVICE)
run = P.run_sam2_video(vp, clip, boxes0.boxes, DEVICE)
print(f"SAM 2: {len(boxes0)} об'єктів, init_state {run.init_ms / 1000:.1f} с, propagate {np.median(run.ms):.0f} мс/кадр")

# ---- покадрово, без пам'яті: промпт = бокс маски з попереднього кадру ----
T = len(clip.frame_ids)
naive_boxes = {o: np.full((T, 4), np.nan, np.float32) for o in range(len(boxes0))}
naive_area = {o: np.zeros(T) for o in range(len(boxes0))}
prev = boxes0.boxes.copy()
t_naive = []
for k in tqdm(range(T), desc="SAM покадрово"):
    fr = clip.read(k)
    with torch.inference_mode(), P.autocast(DEVICE), P.timed(t_naive):
        img_pred.set_image(fr)                                    # енкодер на кожному кадрі
        m, _, _ = img_pred.predict(box=prev, multimask_output=False)
    m = m.reshape(len(prev), H, W) > 0
    for o in range(len(prev)):
        b = P.mask_to_box(m[o])
        naive_area[o][k] = m[o].sum()
        if b is not None:
            naive_boxes[o][k] = b
            prev[o] = b                                           # наступний кадр — від цієї маски
print(f"покадрово: {np.median(t_naive):.0f} мс/кадр (енкодер + декодер на {len(prev)} боксів)")

# ---- зіставлення з GT і метрики ----
M = P.box_iou(boxes0.boxes, gt0)
match = {o: int(g0.tid.values[M[o].argmax()]) for o in range(len(boxes0)) if M[o].max() >= 0.5}
ev_sam2 = {o: P.track_eval(run.boxes[o], gt, t) for o, t in match.items()}
ev_naive = {o: P.track_eval(naive_boxes[o], gt, t) for o, t in match.items()}
cmp = pd.concat({"SAM покадрово": P.summarize_tracks({f"obj {o} / tid {t}": ev_naive[o] for o, t in match.items()}),
                 "SAM 2 з пам'яттю": P.summarize_tracks({f"obj {o} / tid {t}": ev_sam2[o] for o, t in match.items()})})
cmp["мерехтіння, %"] = [P.flicker(naive_area[o]) for o in match] + [P.flicker(run.areas[o]) for o in match]
display(cmp)
print(f"об'єктів без пари в GT (хибні бокси детектора): {len(boxes0) - len(match)}")

In [ ]:
# Площа маски одного об'єкта в часі: найбільший зіставлений об'єкт
o_show = max(match, key=lambda o: (boxes0.boxes[o, 2] - boxes0.boxes[o, 0]) * (boxes0.boxes[o, 3] - boxes0.boxes[o, 1]))
st = ev_sam2[o_show].state.values
fig, ax = plt.subplots(figsize=(10, 3.4))
for k in range(T):
    if st[k] != "видно":
        ax.axvspan(k - .5, k + .5, color="#f2c9b8" if st[k] == "перекрито" else "#e5e5e5", lw=0)
ax.plot(naive_area[o_show], color="#e0703a", lw=1.3, label=f"SAM покадрово · мерехтіння {P.flicker(naive_area[o_show]):.0f} %")
ax.plot(run.areas[o_show], color="#2bb5a0", lw=1.8, label=f"SAM 2 з пам'яттю · мерехтіння {P.flicker(run.areas[o_show]):.0f} %")
ax.set_xlabel("кадр кліпу"); ax.set_ylabel("площа маски, px")
ax.set_title(f"obj {o_show} / GT-трек {match[o_show]} · рожеве — GT «сильно перекрито», сіре — об'єкта вже немає в кадрі")
ax.legend(loc="upper right")
P.save_fig(fig, OUT / "03_area_naive_vs_sam2.png")

**Підсумок розділу.** Без пам'яті маска «дихає» від кадру до кадру, а коли машина ховається за деревами, покадровий SAM
продовжує обводити те, що опинилось у старому боксі — дерева, дорогу, сусіда. SAM 2 тримає ідентичність через пам'ять, а
голова оклюзії здебільшого прибирає маску, коли об'єкта не видно (слайд 28). Покадровий варіант ще й дорожчий: енкодер на
кожному кадрі для кожного кадру — і це без детектора.

## 4. Оклюзія, дрейф і корекція в середині ролика

Пам'ять SAM 2 теж помиляється: якщо в пам'ять потрапила хибна маска, далі модель упевнено веде не той об'єкт (слайд 30).
**SAMURAI** вирішує це моделлю руху: фільтр Калмана прогнозує, де бокс *має* бути, і кадр із великою нев'язкою не
пишеться в пам'ять. Модифікувати пам'ять SAM 2 тут не будемо — використаємо той самий сигнал як **детектор підозрілих
кадрів**: нев'язка = 1 − IoU(прогноз Калмана, бокс маски). Потім на першому такому кадрі «оператор» дає один бокс
(слайд 29) — тут його бере з GT — і SAM 2 перераховує решту ролика.

In [ ]:
def motion_check(boxes, gate=0.3):
    # 1 − IoU між прогнозом Калмана і боксом маски; спостереження з великою нев'язкою в модель руху не приймаємо
    resid = np.full(len(boxes), np.nan)
    kf = None
    for k, b in enumerate(boxes):
        if np.isnan(b[0]):
            if kf is not None:
                kf.predict()
            continue
        if kf is None:
            kf = P.ConstVelKF(b)
            continue
        pred_box = kf.predict()
        resid[k] = 1 - P.box_iou(pred_box, b)[0, 0]
        if resid[k] < 1 - gate:
            kf.update(b)
    return resid


rows = []
for o, t in match.items():
    resid = motion_check(run.boxes[o])
    ev = ev_sam2[o]
    bad = (ev.state == "видно") & (ev.iou < 0.3) | ev.switch | ((ev.state != "видно") & ev.present)
    flag = resid > 0.5
    rows.append({"об'єкт": f"obj {o} / tid {t}", "проблемних кадрів (за GT)": int(bad.sum()),
                 "позначено моделлю руху": int(np.nansum(flag)),
                 "з них справді проблемні": int((flag & bad).sum()),
                 "перший позначений кадр": str(np.flatnonzero(flag)[0]) if np.any(flag) else "—"})
flags = pd.DataFrame(rows).set_index("об'єкт")
flags

In [ ]:
# Корекція: об'єкт із найбільшою кількістю кадрів, де маска не на «своїй» машині, поки ту видно
bad_vis = {o: int(((ev_sam2[o].state == "видно") & ((ev_sam2[o].iou < 0.3) | ev_sam2[o].switch)).sum()) for o in match}
o_fix = max(bad_vis, key=bad_vis.get)
ev = ev_sam2[o_fix]
cand = np.flatnonzero((ev.state.values == "видно") & ((ev.iou.values < 0.3) | ev.switch.values))
if bad_vis[o_fix] == 0:
    print("SAM 2 на цьому кліпі не втратив жодного видимого об'єкта — корекція не потрібна. Спробуйте інший START або SAM2 = 'sam2.1-tiny'.")
else:
    k_fix = int(cand[0])
    g = gt[(gt.k == k_fix) & (gt.tid == match[o_fix])][["x0", "y0", "x1", "y1"]].values[0]
    run_fix = P.run_sam2_video(vp, clip, boxes0.boxes, DEVICE, corrections={k_fix: {o_fix: g}})
    ev_fix = P.track_eval(run_fix.boxes[o_fix], gt, match[o_fix])
    display(pd.concat({"до корекції": P.summarize_tracks({f"obj {o_fix}": ev}),
                       f"один бокс на кадрі {k_fix}": P.summarize_tracks({f"obj {o_fix}": ev_fix})}))
    # кадр для картинки: до корекції маска є там, де об'єкта вже немає / не видно, а після — немає
    gone = np.flatnonzero(ev.present.values & (ev.state.values != "видно") & ~ev_fix.present.values)
    k_show = int(gone[len(gone) // 2]) if len(gone) else min(T - 1, k_fix + 10)
    fr = clip.read(k_show)
    gb = gt[(gt.k == k_show) & (gt.tid == match[o_fix])][["x0", "y0", "x1", "y1"]].values
    centre = run.boxes[o_fix][k_show] if not np.isnan(run.boxes[o_fix][k_show][0]) else (gb[0] if len(gb) else run.boxes[o_fix][k_fix])
    panels = [P.crop_around(P.draw_boxes(P.overlay_masks(fr, [r.small[k_show].get(o_fix, np.zeros((1, 1), bool))], alpha=.6),
                                         gb, color=(150, 150, 160), thick=2), centre, pad=150)[0] for r in (run, run_fix)]
    state = ev.state.values[k_show]
    P.show_row(panels, [f"кадр {k_show} (GT: {state}): до корекції", f"кадр {k_show}: після одного боксу на кадрі {k_fix}"],
               size=5, suptitle="сірий — GT-бокс цього об'єкта, якщо він є", save=OUT / "03_mid_video_correction.png")

**Підсумок розділу.**
* Модель руху ловить більшість кадрів, де маска перескочила або «домальовує» сховану машину, — нуль тренування, кілька
  рядків коду. Це та сама ідея, що в ByteTrack: рух фільтрує вигляд (заняття 9).
* Один бокс у середині ролика — і все після нього перераховується від цієї точки (слайд 29). У конвеєрі розмітки це
  «одна мітка людини на трек» замість малювання кожного кадру.
* Обидва сигнали ламаються на різких маневрах камери: тоді потрібна стабілізація або клік.

## 5. Скільки це коштує

Вартість кадру композиції (слайд 53):
$$T_{frame} \approx \frac{T_{GDINO} + T_{SAM\,enc}}{K} + T_{SAM2\,enc} + n_{obj}\,(T_{mem} + T_{dec})$$
K — крок ключових кадрів (детектор амортизується), $n_{obj}$ — кількість об'єктів. Міряємо доданки на цьому кліпі й
порівнюємо з **«детектор + ByteTrack»** (YOLO11m COCO, класи car / bus / truck), де вартість від $n_{obj}$ не залежить (слайд 31).

In [ ]:
# доданки
t_gd = P.bench(lambda: detect_full(key), n_warmup=1, n_iter=5)["median_ms"]
t_gd_tile = P.bench(lambda: P.detect_tiled(detect_full, key, TILE, OVERLAP), n_warmup=0, n_iter=3)["median_ms"]
def sam_enc():
    with torch.inference_mode(), P.autocast(DEVICE):
        img_pred.set_image(key)


t_enc = P.bench(sam_enc, n_warmup=1, n_iter=5)["median_ms"]

# SAM 2 propagate від кількості об'єктів (для n > знайдених боксів — зсунуті копії, лише для заміру ціни)
rng = np.random.default_rng(SEED)
base = boxes0.boxes
scale_rows = []
for n in [1, 2, 4, 8, 16, 32]:
    reps = np.concatenate([base] * int(np.ceil(n / len(base))))[:n] + rng.normal(0, 2, (n, 4)).astype(np.float32)
    r = P.run_sam2_video(vp, clip, reps, DEVICE, keep_scale=0.05)
    scale_rows.append({"n_obj": n, "SAM 2, мс/кадр": float(np.median(r.ms))})
scale = pd.DataFrame(scale_rows)
slope, icpt = np.polyfit(scale.n_obj, scale["SAM 2, мс/кадр"], 1)       # нахил ≈ T_mem + T_dec на об'єкт

# детектор + ByteTrack: один прохід на кадр, ID з моделі руху
yolo = P.load_ultra("yolo11")
t_bt, n_tracks = [], set()
for k in range(T):
    fr = clip.read(k)
    with P.timed(t_bt):
        r = yolo.track(np.ascontiguousarray(fr[..., ::-1]), persist=True, tracker="bytetrack.yaml", classes=[2, 5, 7],
                       imgsz=1280, conf=0.25, verbose=False)[0]
    if r.boxes.id is not None:
        n_tracks |= set(r.boxes.id.int().tolist())
t_bt = float(np.median(t_bt[3:]))
scale["YOLO11m + ByteTrack, мс/кадр"] = t_bt

nodes = pd.DataFrame([
    {"вузол": "Grounding DINO-B, повний кадр", "мс": t_gd},
    {"вузол": f"Grounding DINO-B, тайли ×{n_pass}", "мс": t_gd_tile},
    {"вузол": "SAM 2.1 image encoder (set_image)", "мс": t_enc},
    {"вузол": "SAM 2 propagate: незалежна від n_obj частина", "мс": icpt},
    {"вузол": "SAM 2 propagate: на кожен об'єкт (T_mem + T_dec)", "мс": slope},
    {"вузол": f"YOLO11m + ByteTrack ({len(n_tracks)} треків)", "мс": t_bt},
])
nodes["FPS"] = 1000 / nodes["мс"]
nodes["залізо"] = GPU
nodes.to_csv(OUT / "bench_pipeline_nodes.csv", index=False)
display(nodes.round(1))

K = 30                                          # ключовий кадр раз на секунду
n = len(boxes0)
T_frame = (t_gd_tile + t_enc) / K + icpt + n * slope
print(f"T_frame(K={K}, n_obj={n}) ≈ ({t_gd_tile:.0f} + {t_enc:.0f}) / {K} + {icpt:.0f} + {n}·{slope:.1f} = {T_frame:.0f} мс "
      f"→ {1000 / T_frame:.0f} FPS")
print(f"виміряно на кліпі: детектор / K = {(t_gd_tile + t_enc) / K:.0f} мс + propagate {np.median(run.ms):.0f} мс "
      f"= {(t_gd_tile + t_enc) / K + np.median(run.ms):.0f} мс")

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.plot(scale.n_obj, scale["SAM 2, мс/кадр"], "o-", color="#2bb5a0", label="SAM 2 propagate (маски + ID)")
ax.plot(scale.n_obj, icpt + slope * scale.n_obj, ":", color="#2bb5a0", lw=1, label=f"лінійна апроксимація: {icpt:.0f} + {slope:.1f}·n")
ax.axhline(t_bt, color="#3a7be0", label="YOLO11m + ByteTrack (бокси + ID)")
ax.axhline(1000 / 30, color="#e0703a", ls="--", lw=1)
ax.text(scale.n_obj.max(), 1000 / 30 * 1.04, "30 FPS", color="#e0703a", ha="right", va="bottom", fontsize=9)
ax.set_xlabel("об'єктів у кадрі, n_obj"); ax.set_ylabel("мс / кадр")
ax.set_title(f"Вартість росте з n_obj лише в SAM 2 · {GPU}")
ax.legend(fontsize=8)
P.save_fig(fig, OUT / "03_cost_vs_nobj.png")

**Підсумок розділу.** Детектор на ключових кадрах амортизується, а SAM 2 — ні: кожен об'єкт додає свою пам'ять і свій
декодер. Для 1–3 об'єктів, які веде оператор, це ок; для автоматичного обліку десятків — «детектор + ByteTrack» з фіксованою
ціною на кадр (слайд 31). Маска — це дорого: беріть її, лише коли бокса справді не вистачає.

## 6. Каркас домашнього завдання: увесь пайплайн однією функцією

**текст → бокси → маски → поширення → експорт** (слайд 63). Функція нижче — стартова точка ДЗ: замініть кліп на свій ролик
і допишіть таблицю «промпт → precision / recall», три провали з каталогу (слайд 51), FPS вузлів і корекцію в середині.
Експорт: `masks/` — PNG з ID об'єкта в кожному пікселі (0 — фон), `cutout/` — об'єкти на приглушеному фоні, `overlay.mp4`,
`annotations.json` — COCO з RLE-масками, `timings.json` — мс по вузлах.

In [ ]:
from pycocotools import mask as mask_utils


def grounded_sam2(clip, prompt, classes, out_dir, box_thr=BOX_THR, tiled=True, corrections=None):
    out_dir = Path(out_dir)
    for sub in ("masks", "cutout"):
        (out_dir / sub).mkdir(parents=True, exist_ok=True)
    tm = {}
    key = clip.read(0)
    # 1. текст → бокси
    det = lambda im: gd(im, prompt, classes, box_thr=box_thr, text_thr=TEXT_THR)
    t = []
    with P.timed(t):
        d = P.detect_tiled(det, key, TILE, OVERLAP)[0] if tiled else det(key)
    tm["detector_ms"] = t[0]
    # 2. фільтр
    d = filter_boxes(d)
    # 3–4. маски + поширення; запис на льоту, щоб не тримати повнороздільні маски в RAM
    W_, H_ = clip.size
    writer = cv2.VideoWriter(str(out_dir / "overlay.mp4"), cv2.VideoWriter_fourcc(*"mp4v"), 15, (W_, H_))
    coco = {"images": [], "annotations": [], "categories": [{"id": i + 1, "name": c} for i, c in enumerate(classes)]}

    def on_frame(k, obj_ids, masks):
        fr = clip.read(k)
        ids = np.zeros((H_, W_), np.uint16)
        for o, m in zip(obj_ids, masks):
            ids[m] = o + 1
            if m.any():
                rle = mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))
                rle["counts"] = rle["counts"].decode()
                coco["annotations"].append({"id": len(coco["annotations"]) + 1, "image_id": k, "track_id": o,
                                            "category_id": int(d.labels[o]) + 1, "segmentation": rle,
                                            "bbox": mask_utils.toBbox(rle).tolist(), "area": float(m.sum()),
                                            "score": float(d.scores[o])})
        coco["images"].append({"id": k, "file_name": clip.frames[k].name, "width": W_, "height": H_,
                               "source_frame": clip.frame_ids[k]})
        cv2.imwrite(str(out_dir / "masks" / f"{k:05d}.png"), ids)
        cut = np.where((ids > 0)[..., None], fr, (fr * 0.15).astype(np.uint8))
        cv2.imwrite(str(out_dir / "cutout" / f"{k:05d}.jpg"), cut[..., ::-1])
        writer.write(P.overlay_masks(fr, list(masks), ids=list(obj_ids))[..., ::-1])

    r = P.run_sam2_video(vp, clip, d.boxes, DEVICE, corrections=corrections, on_frame=on_frame)
    writer.release()
    tm.update({"sam2_init_ms": r.init_ms, "sam2_propagate_ms_median": float(np.median(r.ms)), "n_obj": len(d),
               "frames": len(clip.frame_ids), "hardware": GPU, "prompt": prompt, "box_thr": box_thr})
    (out_dir / "annotations.json").write_text(json.dumps(coco))
    (out_dir / "timings.json").write_text(json.dumps(tm, indent=2, ensure_ascii=False))
    return d, r, tm


EXPORT = OUT / "export"
det_hw, run_hw, tm = grounded_sam2(clip, PROMPT, CLASSES, EXPORT)
print(json.dumps(tm, indent=1, ensure_ascii=False))
print("→", EXPORT, "|", ", ".join(sorted(p.name for p in EXPORT.iterdir())))
ks = [0, T // 3, 2 * T // 3, T - 1]
P.show_row([P.overlay_masks(clip.read(k), [run_hw.small[k][o] for o in run_hw.objs], ids=run_hw.objs) for k in ks],
           [f"кадр {k}" for k in ks], size=4.6, ncols=4, save=OUT / "03_propagation_strip.png")

## 7. Машина розмітки: три кошики і бюджет каналу

Мета конвеєра — людина перевіряє 10–20 % псевдоміток замість того, щоб малювати 100 % (слайд 58). Кожна пара «об'єкт × кадр»
отримує три дешеві сигнали: score детектора на ключовому кадрі, узгодженість із моделлю руху (розділ 4) і стрибок площі
відносно медіани треку. Правила — у клітинці; GT VisDrone показує, наскільки чесно кожен кошик.

In [ ]:
lab_rows = []
for o in run.objs:
    resid = motion_check(run.boxes[o])
    med_area = np.median(run.areas[o][run.areas[o] > 0]) if (run.areas[o] > 0).any() else 1
    ev = ev_sam2.get(o)
    for k in range(T):
        if run.areas[o][k] == 0:
            continue                                                       # маски немає — мітки немає
        ok = ev is not None and ev.state[k] == "видно" and ev.iou[k] >= 0.5 and not ev.switch[k]
        lab_rows.append(dict(obj=o, k=k, score=boxes0.scores[o], motion=1 - np.nan_to_num(resid[k], nan=0.0),
                             area_ratio=run.areas[o][k] / med_area, correct=ok))
lab = pd.DataFrame(lab_rows)


def route(r):
    if r.score < 0.35 or r.motion < 0.2 or not 0.25 <= r.area_ratio <= 4:
        return "відкинути"
    if r.score >= 0.45 and r.motion >= 0.6 and 0.6 <= r.area_ratio <= 1.6:
        return "прийняти"
    return "перевірити"


lab["кошик"] = lab.apply(route, axis=1)
baskets = lab.groupby("кошик").agg(міток=("correct", "size"), **{"правильних за GT": ("correct", "mean")})
baskets["частка міток"] = baskets["міток"] / len(lab)
display(baskets.loc[[b for b in ["прийняти", "перевірити", "відкинути"] if b in baskets.index]]
        .style.format({"правильних за GT": "{:.0%}", "частка міток": "{:.0%}"}))

# Калібрування автоприйняття: що дає кожен сигнал. Score детектора — один на весь трек (ключовий кадр),
# тож дрейф маски в середині ролика бачать лише покадрові сигнали — рух і площа.
sweep = []
for s_thr in [0.3, 0.4, 0.5, 0.6]:
    for name, extra in [("лише score", np.ones(len(lab), bool)),
                        ("score + рух + площа", (lab.motion >= 0.6) & lab.area_ratio.between(0.6, 1.6))]:
        acc_m = (lab.score >= s_thr) & extra
        sweep.append({"score ≥": s_thr, "правило": name, "автоприйнято": f"{acc_m.mean():.0%}",
                      "помилкових міток серед автоприйнятих": int((~lab.correct[acc_m]).sum())})
display(pd.DataFrame(sweep).pivot(index="score ≥", columns="правило",
                                  values=["автоприйнято", "помилкових міток серед автоприйнятих"]))
acc = lab[lab["кошик"] == "прийняти"]
audit = acc.sample(frac=0.05, random_state=SEED) if len(acc) else acc
print(f"людина дивиться: «перевірити» ({(lab['кошик'] == 'перевірити').mean():.0%}) + 5 % «прийняти» на аудит "
      f"({len(audit)} міток); помилок, що пройдуть у датасет без перегляду: {int((~acc.correct).sum())} з {len(acc)}")

In [ ]:
# Бюджет каналу (слайд 60): скільки байтів на один об'єкт, якщо передавати не бокс, а маску
k_mid = T // 2
ids = cv2.imread(str(EXPORT / "masks" / f"{k_mid:05d}.png"), cv2.IMREAD_UNCHANGED)
rows = []
for o in np.unique(ids[ids > 0]):
    m = ids == o
    cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    poly = cv2.approxPolyDP(max(cs, key=cv2.contourArea), 1.5, True)          # полігон із допуском 1.5 px
    runs = P.rle_encode(m)
    coco_rle = mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))["counts"]
    rows.append({"бокс: тип + 4 × uint16": 1 + 4 * 2, "полігон: вершини × 2 × uint16": len(poly) * 4,
                 "RLE, varint": P.varint_bytes(runs), "COCO RLE (рядок)": len(coco_rle),
                 f"бітова маска {W}×{H}": W * H // 8})
budget = pd.DataFrame(rows).median().rename("байтів на об'єкт, медіана").to_frame()
budget["× до боксу"] = budget.iloc[:, 0] / budget.iloc[0, 0]
display(budget.round(1))

**Підсумок розділу.**
* Кошики не роблять розмітку безкоштовною: «прийняти» з аудитом 5 % — це те, що піде в датасет майже без людини, і його
  точність за GT — головна цифра конвеєра («частка псевдоміток без правок», слайд 50). Пороги не вгадують, а калібрують
  на своєму домені: розгортка показує, що score детектора сам по собі пропускає дрейф маски в датасет, а дешеві покадрові
  сигнали (рух, площа) його відсікають. Кошик «відкинути» теж має ціну — частина викинутих міток правильна. Систематична помилка промпта
  потрапляє в кожен кадр, а пропущені детектором машини не видно без аналізу пропусків (заняття 13).
* Маска в каналі коштує в рази дорожче за бокс: полігон — ×10, RLE — десятки разів, бітова маска — десятки тисяч.
  Якщо оператору досить типу й координат — передавайте бокс (слайд 60).

## 8. ІЧ: геометрія переживає домен, семантика — ні

Ілюстративна пара RGB / LWIR із заняття 1 (кадри не зареєстровані попіксельно, тож порівнюємо кожен кадр окремо).
**Текстовий промпт:** Grounding DINO «tank» на RGB, на LWIR і на LWIR із псевдокольором. **Геометричний промпт:** точки на
танках LWIR — їх дає оператор або детектор руху / гарячих плям (слайд 47) — і SAM 2.1 будує маски.

In [ ]:
rgb = P.imread_rgb(P.IMAGES_DIR / "lesson_1_demo_1_rgb.jpg")
lwir = P.imread_rgb(P.IMAGES_DIR / "lesson_1_demo_1_lwir.jpg")
CROP = (1950, 750, 2650, 1400)                         # ділянка з технікою, px кадру 2752 × 1536
x0, y0, x1, y1 = CROP
rgb_c, lwir_c = rgb[y0:y1, x0:x1], lwir[y0:y1, x0:x1]
pseudo = cv2.applyColorMap(cv2.cvtColor(lwir_c, cv2.COLOR_RGB2GRAY), cv2.COLORMAP_INFERNO)[..., ::-1]

rows, vis = [], []
for name, im in [("RGB", rgb_c), ("LWIR", lwir_c), ("LWIR + псевдоколір", pseudo)]:
    d = gd(im, "tank . armored vehicle .", box_thr=0.3)
    rows.append({"кадр": name, "боксів «tank»": len(d), "макс. score": float(d.scores.max()) if len(d) else 0.0})
    vis.append(P.draw_boxes(im, d.boxes, [f"{s:.2f}" for s in d.scores]))
display(pd.DataFrame(rows).set_index("кадр").round(2))

# геометричний промпт: по точці в кожну машину на LWIR (координати кропу)
LWIR_POINTS = np.array([[155, 95], [310, 135], [270, 230], [245, 350], [380, 485], [500, 195], [450, 128]], np.float32)
with torch.inference_mode(), P.autocast(DEVICE):
    img_pred.set_image(lwir_c)
    ms_ir, iou_ir = [], []
    for pt in LWIR_POINTS:
        m, i, _ = img_pred.predict(point_coords=pt[None], point_labels=np.array([1]), multimask_output=True)
        ms_ir.append(m[int(np.argmax(i))] > 0)
        iou_ir.append(float(i.max()))
seg = P.overlay_masks(lwir_c, ms_ir, alpha=.55)
for x, y in LWIR_POINTS:
    cv2.circle(seg, (int(x), int(y)), 5, (255, 60, 60), -1)
print(f"SAM 2.1 на LWIR, клік у кожну машину: predicted_iou {np.round(iou_ir, 2)}")
P.show_row(vis + [seg], ["RGB: текст «tank»", "LWIR: текст «tank»", "LWIR + псевдоколір: текст", "LWIR: клік → SAM 2.1"],
           size=4, ncols=4, save=OUT / "03_ir_text_vs_geometry.png")

**Підсумок розділу.** На тепловізорі словник «tank» майже не працює (тепловізійних кадрів у вебовому претренуванні
майже немає), а псевдоколір — косметика: домен він не переносить. Геометрія ж виживає — SAM 2.1 обводить машини за кліком.
Звідси робоча стратегія для ІЧ: **геометричний промпт** (рух, гарячі плями, RGB-канал після реєстрації) → SAM 2 веде маски →
людина ставить одну мітку на трек (слайд 58). Повне злиття RGB + IR — заняття 12; бонус ДЗ — те саме на HIT-UAV.

## 9. Композиція чи одна модель: SAM 3

SAM 3 приймає **поняття** (коротку фразу) і сам повертає всі екземпляри з масками — 4 вузли Grounded-SAM в одній моделі
(слайди 32, 54). Порівнюємо на ключовому кадрі з Grounding DINO + SAM 2.1: точність за GT, час, і чи є куди вставити
власний фільтр. Ліцензія SAM 3 — SAM License, не Apache: для продукту й оборонного застосування — лише після юридичного аудиту.

In [ ]:
if not USE_SAM3:
    print("SAM 3 немає в artifacts/models/sam — розділ пропущено. Ваги gated: прийміть ліцензію на "
          "https://huggingface.co/facebook/sam3, `hf auth login`, потім `python tools/fetch_sam_models.py --models sam3`.")
else:
    import PIL

    del vp
    P.free_memory()
    s3_proc, s3 = P.load_hf("sam3", DEVICE)
    inp = s3_proc(images=PIL.Image.fromarray(key), text="car", return_tensors="pt").to(DEVICE)

    def sam3_run():
        with torch.inference_mode():
            out = s3(**inp)
        return s3_proc.post_process_instance_segmentation(out, threshold=0.5, mask_threshold=0.5,
                                                          target_sizes=inp["original_sizes"].tolist())[0]

    r3 = sam3_run()
    t3 = P.bench(sam3_run, n_warmup=1, n_iter=5)["median_ms"]

    def composition():
        d = filter_boxes(P.detect_tiled(detect_full, key, TILE, OVERLAP)[0])
        with torch.inference_mode(), P.autocast(DEVICE):
            img_pred.set_image(key)
            m, _, _ = img_pred.predict(box=d.boxes, multimask_output=False)
        return d, m.reshape(len(d), H, W) > 0

    d_c, m_c = composition()
    t_c = P.bench(composition, n_warmup=0, n_iter=3)["median_ms"]
    b3 = r3["boxes"].float().cpu().numpy()
    rows = []
    for name, b, t in [("Grounding DINO-B (тайли) + фільтр + SAM 2.1", d_c.boxes, t_c), ("SAM 3, «car»", b3, t3)]:
        pr = P.match_pr(b, gt0)
        rows.append({"підхід": name, "об'єктів": len(b), "TP": pr["tp"], "FP": pr["fp"], "пропущено": pr["fn"], "мс": t})
    display(pd.DataFrame(rows).set_index("підхід"))
    P.show_row([P.overlay_masks(key, m_c), P.overlay_masks(key, r3["masks"].cpu().numpy())],
               ["Grounding DINO + SAM 2.1", "SAM 3"], size=7, save=OUT / "03_sam3_vs_composition.png")
    del s3
    P.free_memory()

**Підсумок розділу.** SAM 3 закриває весь ланцюг одним проходом і часто знаходить більше, але це один непрозорий вихід:
немає місця для власного фільтра між «знайти» й «обвести», не видно, який вузол помилився, і інша ліцензія (слайд 54).
Для конвеєра розмітки, де помилки треба розбирати повузлово, композиція зазвичай виграє; для швидкого прототипу — SAM 3.

**Питання в аудиторію:** за якого `n_obj` і якого кроку ключових кадрів K композиція перестає вкладатися в 30 FPS на
вашому залізі — порахуйте за формулою розділу 5 і перевірте на графіку.